In [1]:
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from sqlalchemy import create_engine


def get_kospi_stocks(pages=4):
    base_url = "https://finance.naver.com/sise/sise_market_sum.naver?&page="
    all_data = []

    for page in range(1, pages + 1):
        url = base_url + str(page)
        tables = pd.read_html(url, encoding='euc-kr')
        df = tables[1].dropna(how='all')  # 불필요한 빈 행 제거
        all_data.append(df)
        time.sleep(1)  # 서버 과부하 방지

    combined = pd.concat(all_data)
    combined = combined.reset_index(drop=True)
    return combined

kospi_df = get_kospi_stocks()

kospi_df.columns = [col.strip() for col in kospi_df.columns]
kospi_df.rename(columns={"종목명": "stock_name", "현재가": "current_price", "시가총액": "market_cap"}, inplace=True)

engine = create_engine("sqlite:///stocks.db")  # 같은 디렉토리에 stocks.db 파일 생성
kospi_df.to_sql("kospi_stocks", con=engine, if_exists="replace", index=False)

pd.read_sql("SELECT stock_name, current_price, market_cap FROM kospi_stocks LIMIT 10", con=engine)

ImportError: Missing optional dependency 'lxml'.  Use pip or conda to install lxml.